# 📓 Notebook 02 — RAG Chain & Conversational Q&A

**Project:** IncidentIQ — AI-powered Incident Intelligence for Firefighters

**Goal of this notebook:** Build a conversational question-answering chain on top of the ChromaDB knowledge base created in Notebook 01.

## What this notebook covers
1. Load the existing ChromaDB vectorstore
2. Define the system prompt — language rules, citation rules, tone
3. Build the retriever — fetch relevant chunks per question
4. Add conversation memory — agent remembers previous questions
5. Build the full RAG chain using LangChain LCEL
6. Test the chain with English and Dutch questions
7. Run quality tests on the RAG chain

## Why this matters
Notebook 01 built the knowledge base. This notebook builds the **brain** on top of it — the chain that retrieves relevant context and generates accurate, cited answers in any language.

---

## 1. Install required packages

All packages were installed in Notebook 01.
Run this cell only if you are starting fresh in a new environment.

In [ ]:
!pip install langchain langchain-openai langchain-community chromadb python-dotenv -q
print("✅ Packages ready.")

## 2. Import libraries and load environment variables

We import all LangChain components needed to build the RAG chain.
The OpenAI API key is loaded from the .env file — same as Notebook 01.

In [ ]:
import os
import re
from dotenv import load_dotenv

# LangChain core
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema import HumanMessage, AIMessage
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# ChromaDB
import chromadb

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "❌ OPENAI_API_KEY not found. Check your .env file."
print("✅ Environment variables loaded successfully.")

## 3. Configuration

Central configuration for the RAG chain.
CHROMA_PATH and COLLECTION_NAME must match the values used in Notebook 01.

In [ ]:
# ── Must match Notebook 01 settings ──────────────────────────────────────────
CHROMA_PATH      = "../chroma_db"
COLLECTION_NAME  = "incidentiq"
EMBEDDING_MODEL  = "text-embedding-3-small"

# ── LLM settings ──────────────────────────────────────────────────────────────
LLM_MODEL        = "gpt-4o-mini"   # Cost-efficient model — good for RAG Q&A
LLM_TEMPERATURE  = 0               # Temperature 0 = consistent, factual answers

# ── Retriever settings ────────────────────────────────────────────────────────
# k = number of chunks retrieved per question
# Higher k = more context but higher cost and slower response
RETRIEVER_K      = 5

print(f"✅ Configuration set.")
print(f"   ChromaDB path:   {CHROMA_PATH}")
print(f"   Collection:      {COLLECTION_NAME}")
print(f"   LLM model:       {LLM_MODEL}")
print(f"   Retriever k:     {RETRIEVER_K}")

## 4. Load ChromaDB vectorstore

We load the existing ChromaDB collection created in Notebook 01.
No new embeddings are generated here — we reuse what was already stored.

In [ ]:
# Initialize embedding model — must be the same model used in Notebook 01
# Using a different model would produce incompatible vectors
embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Load existing ChromaDB collection from disk
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
vectorstore = Chroma(
    client=chroma_client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
)

# Verify the collection contains data from Notebook 01
chunk_count = chroma_client.get_collection(COLLECTION_NAME).count()
assert chunk_count > 0, "❌ ChromaDB is empty. Run Notebook 01 first."

print(f"✅ ChromaDB loaded successfully.")
print(f"   Collection: '{COLLECTION_NAME}'")
print(f"   Chunks available: {chunk_count}")

## 5. Define the system prompt

The system prompt controls the agent's behavior, language rules, citation style and tone.
This is the single most important configuration for answer quality.

**Key rules we enforce:**
- Answer only from the retrieved context — no hallucination
- Always respond in the same language as the user's question
- Always cite the timestamp where the information was found
- Say clearly when the answer is not in the context

In [ ]:
SYSTEM_PROMPT = """You are IncidentIQ, an AI assistant specialized in firefighter training and incident analysis.
You help firefighters and commanders extract knowledge from incident videos.

ANSWER RULES:
- Answer strictly based on the retrieved context below.
- Never make up information that is not present in the context.
- If the answer is not in the context, say: "I could not find this information in the video."

LANGUAGE RULE:
- Always respond in the same language as the user's question.
- User asks in English  → answer in English
- User asks in Dutch    → answer in Dutch
- User asks in French   → answer in French

CITATION RULE:
- Always reference the timestamp where the information was found.
- Format: "According to the video at [MM:SS], ..."
- If multiple timestamps are relevant, cite all of them.

TONE:
- Professional, concise and structured.
- Use bullet points for lists of lessons, steps or recommendations.
- Keep answers focused — do not repeat the question.

Context:
{context}

Conversation history:
{chat_history}
"""

print("✅ System prompt defined.")
print(f"   Length: {len(SYSTEM_PROMPT)} characters")

## 6. Build the retriever

The retriever fetches the most relevant chunks from ChromaDB for each question.
We use similarity search with a configurable k value.

We also add a cleaning step that removes timestamp markers from the context
before passing it to the LLM — timestamps are preserved in the raw chunks
for citation purposes but would clutter the LLM context if left in.

In [ ]:
def clean_context(text: str) -> str:
    """
    Remove [MM:SS] timestamp markers from retrieved chunks
    before passing context to the LLM.
    Timestamps are used internally for citation but clutter the LLM input.
    """
    text = re.sub(r'\[\d{2}:\d{2}\]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def retrieve_context(question: str, vectorstore: Chroma, k: int = RETRIEVER_K) -> tuple:
    """
    Retrieve the top-k most relevant chunks for a question.
    Returns (clean_context_for_llm, raw_chunks_with_timestamps).
    """
    results = vectorstore.similarity_search(question, k=k)

    # Clean context for LLM — no timestamps
    clean = clean_context("\n\n".join([r.page_content for r in results]))

    # Raw chunks — kept for citation display
    raw = results

    return clean, raw


# Test the retriever with a sample question
test_context, test_chunks = retrieve_context("What went wrong during the incident?")

print(f"✅ Retriever working correctly.")
print(f"   Retrieved {len(test_chunks)} chunks for test question.")
print(f"   Context length sent to LLM: {len(test_context)} characters")
print(f"\n📄 Context preview (first 300 chars):")
print("-" * 60)
print(test_context[:300])

## 7. Build conversation memory

The agent needs to remember previous questions and answers within a session.
This allows follow-up questions like "Can you elaborate on that?" or
"What did you mean by the stack effect you mentioned earlier?"

We use a simple in-memory list of HumanMessage and AIMessage objects.
This list is passed to the LLM with every new question as conversation history.

In [ ]:
# Conversation history — stores all messages in the current session
# Each entry is either a HumanMessage (user) or AIMessage (agent)
chat_history = []

def format_chat_history(history: list) -> str:
    """
    Format the chat history as a readable string for the system prompt.
    Returns an empty string if no history exists yet.
    """
    if not history:
        return "No previous conversation."
    formatted = []
    for msg in history:
        if isinstance(msg, HumanMessage):
            formatted.append(f"User: {msg.content}")
        elif isinstance(msg, AIMessage):
            formatted.append(f"Assistant: {msg.content}")
    return "\n".join(formatted)


def add_to_history(history: list, question: str, answer: str) -> list:
    """
    Add a question-answer pair to the conversation history.
    Keeps the last 10 exchanges to avoid exceeding the LLM context window.
    """
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=answer))
    return history[-20:]  # Keep last 10 exchanges (20 messages)


print("✅ Conversation memory initialized.")
print(f"   Current history: {len(chat_history)} messages")

## 8. Build the RAG chain

We assemble all components into a single ask() function that:
1. Retrieves relevant chunks from ChromaDB
2. Formats the system prompt with context and chat history
3. Calls the LLM to generate an answer
4. Saves the exchange to memory
5. Returns the answer

This function is the core of the IncidentIQ agent.

In [ ]:
# Initialize the LLM
llm = ChatOpenAI(model=LLM_MODEL, temperature=LLM_TEMPERATURE)


def ask(question: str, verbose: bool = False) -> str:
    """
    Ask a question about the loaded video.
    Retrieves relevant context from ChromaDB, formats the prompt,
    calls the LLM and returns the answer in the user's language.

    Args:
        question: The user's question in any language
        verbose:  If True, show retrieved chunks for debugging

    Returns:
        The agent's answer as a string
    """
    global chat_history

    # Step 1 — Retrieve relevant chunks from ChromaDB
    context, raw_chunks = retrieve_context(question)

    if verbose:
        print(f"\n🔍 Retrieved {len(raw_chunks)} chunks:")
        for i, chunk in enumerate(raw_chunks):
            print(f"  [{i+1}] {chunk.page_content[:100]}...")

    # Step 2 — Format the full prompt with context and history
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "{question}"),
    ])

    # Step 3 — Build and invoke the chain
    chain = prompt | llm | StrOutputParser()

    answer = chain.invoke({
        "context": context,
        "chat_history": format_chat_history(chat_history),
        "question": question,
    })

    # Step 4 — Save to memory
    chat_history = add_to_history(chat_history, question, answer)

    return answer


print("✅ RAG chain ready.")
print("   Use ask('your question here') to query the video.")

## 9. Test the chain — English questions

We test the full RAG chain with English questions about the Karel Lambert video.
The agent should answer in English and cite relevant timestamps.

In [ ]:
# Reset conversation history for a clean test
chat_history = []

print("=" * 60)
print("🧪 TEST — English Q&A")
print("=" * 60)

# Question 1 — General topic
q1 = "What is the main topic of this video?"
print(f"\n💬 Q: {q1}")
a1 = ask(q1)
print(f"🤖 A: {a1}")

# Question 2 — Specific incident detail
q2 = "What went wrong during the incident?"
print(f"\n💬 Q: {q2}")
a2 = ask(q2)
print(f"🤖 A: {a2}")

# Question 3 — Follow-up question using memory
q3 = "Can you give more detail about the first problem you mentioned?"
print(f"\n💬 Q: {q3}")
a3 = ask(q3)
print(f"🤖 A: {a3}")

print(f"\n✅ English test complete — {len(chat_history)//2} exchanges in memory.")

## 10. Test the chain — Dutch questions

The same chain handles Dutch questions automatically.
No translation tool needed — the LLM detects the language and responds accordingly.
The underlying ChromaDB context is always in English (the video language).

In [ ]:
# Reset conversation history for Dutch test
chat_history = []

print("=" * 60)
print("🧪 TEST — Nederlandse Q&A")
print("=" * 60)

# Vraag 1 — Algemeen onderwerp
q1 = "Waarover gaat deze video?"
print(f"\n💬 Vraag: {q1}")
a1 = ask(q1)
print(f"🤖 Antwoord: {a1}")

# Vraag 2 — Specifiek detail
q2 = "Welke fouten werden gemaakt tijdens het incident?"
print(f"\n💬 Vraag: {q2}")
a2 = ask(q2)
print(f"🤖 Antwoord: {a2}")

# Vraag 3 — Vervolgvraag met memory
q3 = "Wat had men beter kunnen doen?"
print(f"\n💬 Vraag: {q3}")
a3 = ask(q3)
print(f"🤖 Antwoord: {a3}")

print(f"\n✅ Nederlandse test compleet — {len(chat_history)//2} vragen beantwoord.")

## 11. RAG chain quality tests

Automated tests to verify the chain works correctly before moving to Notebook 03.
Tests cover language detection, memory, citation format and answer quality.

In [ ]:
print("=" * 60)
print("🧪 RAG CHAIN QUALITY TESTS")
print("=" * 60)

tests_passed = 0
tests_failed = 0

def check(name: str, condition: bool, detail: str = ""):
    global tests_passed, tests_failed
    if condition:
        tests_passed += 1
        print(f"  ✅ PASS — {name}")
    else:
        tests_failed += 1
        print(f"  ❌ FAIL — {name}")
    if detail:
        print(f"         {detail}")

# Fresh history for tests
chat_history = []

# ── Test 1: English answer for English question ───────────────────────────────
# The agent must detect English and respond in English
answer_en = ask("What is this video about?")
is_english = any(w in answer_en.lower() for w in ["fire", "building", "incident", "brussels", "the", "and"])
check(
    "English question returns English answer",
    is_english,
    f"Answer preview: {answer_en[:80]}..."
)

# ── Test 2: Dutch answer for Dutch question ───────────────────────────────────
# The agent must detect Dutch and respond in Dutch
chat_history = []
answer_nl = ask("Waarover gaat deze video?")
is_dutch = any(w in answer_nl.lower() for w in ["brand", "gebouw", "brussel", "de", "het", "een", "was", "werd"])
check(
    "Dutch question returns Dutch answer",
    is_dutch,
    f"Answer preview: {answer_nl[:80]}..."
)

# ── Test 3: Answer is long enough to be meaningful ───────────────────────────
check(
    "Answer contains enough content to be useful",
    len(answer_nl) > 100,
    f"Answer length: {len(answer_nl)} characters — minimum: 100"
)

# ── Test 4: Memory works across follow-up questions ───────────────────────────
# Ask a follow-up that only makes sense if the agent remembers the first answer
chat_history = []
ask("What problems occurred during the incident?")
followup = ask("Can you elaborate on the first problem you mentioned?")
memory_works = len(followup) > 50 and "not find" not in followup.lower()
check(
    "Conversation memory works for follow-up questions",
    memory_works,
    f"Follow-up answer length: {len(followup)} characters"
)

# ── Test 5: Agent does not hallucinate on unknown topic ───────────────────────
# Ask about something clearly not in the video
chat_history = []
out_of_scope = ask("What is the boiling point of water?")
handles_unknown = any(phrase in out_of_scope.lower() for phrase in [
    "not find", "not in", "geen", "niet gevonden", "video", "context"
])
check(
    "Agent correctly handles out-of-scope questions",
    handles_unknown,
    f"Answer: {out_of_scope[:100]}..."
)

# ── Test 6: Citation format present in answers ────────────────────────────────
chat_history = []
cited_answer = ask("When did Karel Lambert talk about the command post?")
has_citation = bool(re.search(r'\[\d{2}:\d{2}\]', cited_answer))
check(
    "Answer contains timestamp citation [MM:SS]",
    has_citation,
    f"Answer preview: {cited_answer[:100]}..."
)

# ── Test 7: History is limited to avoid context overflow ─────────────────────
chat_history = []
for i in range(12):
    ask(f"Question number {i+1}: What happened in Brussels?")
check(
    "Chat history stays within 20 messages (10 exchanges)",
    len(chat_history) <= 20,
    f"Current history length: {len(chat_history)} messages"
)

# ── Final results ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"📊 RESULTS: {tests_passed} passed | {tests_failed} failed")
if tests_failed == 0:
    print("✅ All tests passed — RAG chain is ready for Notebook 03!")
else:
    print("⚠️  Fix the failing tests before moving to Notebook 03.")
print("=" * 60)

---

## ✅ What we built in this notebook

| Step | What | Why |
|------|------|-----|
| 1 | Loaded ChromaDB vectorstore | Reuses knowledge base from Notebook 01 |
| 2 | Defined system prompt | Controls language, citation and tone |
| 3 | Built retriever with context cleaner | Removes timestamp noise before LLM input |
| 4 | Added conversation memory | Agent remembers previous questions |
| 5 | Built ask() function | Single entry point for all Q&A |
| 6 | Tested English Q&A | Verified English retrieval and answers |
| 7 | Tested Dutch Q&A | Verified automatic language detection |
| 8 | Ran 7 quality tests | Confirmed chain works correctly end-to-end |

## ➡️ Next: Notebook 03 — Agent Tools
Build the LangChain tools that the LangGraph agent will use:
YouTube transcript tool, RAG retrieval tool, PDF cheatsheet tool and Gmail tool.